# Final experiment notebook: joint multi-label vs. one-vs-rest with BCE and weighted BCE

This notebook consolidates the final experimental comparison for the thesis project and removes the more computationally expensive XLM-RoBERTa branch in order to focus exclusively on `xlm-roberta-base`. The notebook evaluates two modeling formulations and two loss configurations in a unified framework:

- **Joint multi-label classification** with one shared classifier head for all labels.
- **One-vs-Rest classification** with one binary classifier per label.
- **Simple BCEWithLogitsLoss**.
- **Weighted BCEWithLogitsLoss** using class-dependent positive weights.

The workflow is designed so that the final comparison covers the main modeling decision space without opening another large round of exploratory experiments. The notebook therefore aims at three outcomes:

1. A robust model comparison using cross-validation.
2. A dashboard-oriented thresholding strategy in addition to F1-oriented thresholding.
3. Export of the overall best final dashboard model after model selection.

The notebook also includes detailed markdown explanations so that the experimental logic remains transparent when reused for thesis documentation and later implementation work.

## Experimental rationale

The earlier workflow relied on a single train/validation split and a single modeling formulation. That approach was sufficient for feasibility testing, but it remained sensitive to sample composition, especially because several labels are comparatively rare and some threshold values were driven to very low levels. In such a setting, strong recall values can be produced together with low precision, which is undesirable for a dashboard setting where topic counts should remain interpretable.

This final notebook therefore introduces four methodological refinements.

### 1. Reduced model scope

The XLM-RoBERTa model is removed entirely. The earlier comparison suggested that the German sentiment BERT model was competitive while also being easier to work with computationally. Since the goal of this notebook is final selection rather than broad architecture search, the smaller scope is methodologically justified.

### 2. Modeling formulation comparison

The topic classification problem can be treated either as one joint multi-label task or as a set of independent binary decisions. The joint formulation may benefit from shared internal representations across labels, while the One-vs-Rest formulation may provide cleaner decision boundaries for difficult or weak labels. Implementing both approaches inside the same evaluation framework allows a final comparison that is still compact enough to run.

### 3. Loss-function comparison

The original workflow effectively used unweighted BCE. That serves as a proper baseline, but the label distribution is imbalanced. Weighted BCE therefore becomes a relevant final comparison point because it changes the optimization pressure placed on rare positive labels. The present notebook implements both simple BCE and weighted BCE so that the effect of class-imbalance handling can be observed directly.

### 4. More robust evaluation

A single validation split may overstate or understate performance depending on which reviews land in validation. K-fold cross-validation reduces this dependency and produces a more defensible estimate of expected performance. For the final thesis experiment, this is preferable to another round of ad-hoc split tuning.

In [ ]:
import os
import json
import copy
import math
import shutil
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset
from sklearn.model_selection import KFold
from sklearn.metrics import (
    f1_score,
    classification_report,
    precision_recall_fscore_support,
    hamming_loss,
)
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_rows", 100)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Device:", device)

## Configuration

The configuration below defines one backbone model, two modeling approaches, two loss modes, and one cross-validation setup. The default number of folds is set to 5 because this provides a stronger estimate than a single split while remaining tractable. If runtime becomes a serious bottleneck during debugging, the fold count can temporarily be reduced before the final full run is executed.

The dashboard threshold mode is designed to move away from pure F1 maximization. Instead of preferring very low thresholds that often inflate recall, the threshold search first seeks a minimum precision level and only then maximizes recall among acceptable candidates. This aligns the prediction behavior more closely with a dashboard context, where false positives can distort topic counts and downstream interpretation.

In [ ]:
# =========================
# CONFIG
# =========================

LABELED_PATH = "../data/labeling/manual_labeling_pool_hek_viactiv.xlsx"
OUTPUT_DIR = Path("output/03_model_training/results_approach_loss_experiment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_CANDIDATES = ["review_text", "review", "content", "text"]
TOPIC_COL = "label_topics_raw"
ID_CANDIDATES = ["review_id", "id", "app_review_id"]

# --- Use the superior XLM-RoBERTa model ---
MODEL_NAME = "xlm-roberta-base" 

APPROACHES = ["joint_multilabel", "one_vs_rest"]
LOSS_MODES = ["bce", "weighted_bce"]

MAX_LENGTH = 256
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
NUM_EPOCHS = 3
NUM_FOLDS = 3
THRESHOLD_GRID = np.arange(0.05, 0.96, 0.05)
DEFAULT_THRESHOLD = 0.50

# --- Standardize the precision floor ---
DASHBOARD_MIN_PRECISION = 0.50 

FINAL_SELECTION_METRIC = "macro_f1"
FINAL_SELECTION_THRESHOLD_MODE = "dashboard"

SAVE_FOLD_MODELS = False
EXPORT_FINAL_BEST_MODEL = True

## Helper functions

The following helper functions implement reusable logic for column detection, text and topic normalization, dataset construction, weighted loss calculation, threshold selection, and metric computation. Keeping this logic modular makes the experimental workflow easier to inspect and reduces duplication across the different modeling branches.

In [ ]:
def find_first_existing_column(df, candidates, required=True):
    """
    Return the first column name from a candidate list that exists in a DataFrame.

    This helper is used to make the notebook more robust to small schema changes
    across intermediate files, for example when the review text or ID column has
    slightly different names in different exports.
    """
    for col in candidates:
        if col in df.columns:
            return col
    if required:
        raise ValueError(f"None of these columns were found: {candidates}")
    return None


def clean_topic_string(x):
    """
    Convert a raw topic cell into a stripped string representation.

    Missing values are mapped to an empty string so downstream topic splitting
    can be handled consistently without special-case logic.
    """
    if pd.isna(x):
        return ""
    return str(x).strip()


def split_topics(x):
    """
    Split a semicolon-separated topic string into a cleaned list of labels.
    """
    x = clean_topic_string(x)
    if not x:
        return []
    return [t.strip() for t in x.split(";") if str(t).strip()]


def normalize_text(x):
    """
    Normalize review text values into stripped strings and replace missing values with empty text.
    """
    if pd.isna(x):
        return ""
    return str(x).strip()


def build_multihot(topics, label2id):
    """
    Convert a list of topic labels into a multi-hot target vector.
    """
    y = np.zeros(len(label2id), dtype=np.float32)
    for t in topics:
        if t in label2id:
            y[label2id[t]] = 1.0
    return y


class MultiLabelTextDataset(Dataset):
    """
    PyTorch dataset for tokenized text classification with multi-label targets.

    Each dataset item contains tokenized review text plus a float label tensor
    that is compatible with BCE-based multi-label training.
    """
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Tokenize one review on demand so memory usage stays manageable during training.
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item


class BCETrainer(Trainer):
    """
    Custom Hugging Face Trainer that uses BCEWithLogitsLoss for binary or multi-label tasks.

    The trainer optionally accepts positive class weights so that weighted BCE
    can be used for imbalanced label distributions.
    """
    def __init__(self, pos_weight=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if logits.ndim == 1:
            logits = logits.unsqueeze(-1)
        if labels.ndim == 1:
            labels = labels.unsqueeze(-1)
        # Move the positive class weights onto the same device as the logits before computing weighted BCE.
        pos_weight = self.pos_weight.to(logits.device) if self.pos_weight is not None else None
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


def make_joint_model(model_name, num_labels, id2label, label2id):
    """
    Create one Transformer model with a shared classifier head for all topic labels.
    """
    config = AutoConfig.from_pretrained(model_name)
    config.num_labels = num_labels
    config.problem_type = "multi_label_classification"
    config.id2label = id2label
    config.label2id = label2id
    # Load the pretrained backbone and reinitialize the classifier head for the task-specific label space.
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config,
        ignore_mismatched_sizes=True,
    )
    return model


def make_binary_model(model_name, label_name):
    """
    Create a binary Transformer classifier for one label in the One-vs-Rest setup.
    """
    config = AutoConfig.from_pretrained(model_name)
    # Each One-vs-Rest model predicts exactly one label, so the classifier head only needs one output unit.
    config.num_labels = 1
    config.problem_type = "multi_label_classification"
    config.id2label = {0: label_name}
    config.label2id = {label_name: 0}
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config,
        ignore_mismatched_sizes=True,
    )
    return model


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def compute_pos_weight(y_train):
    """
    Compute positive class weights for weighted BCE in the joint multi-label setting.

    The weight for each label is based on the ratio of negative to positive
    training examples so that rarer positive labels receive stronger emphasis.
    """
    pos = y_train.sum(axis=0)
    neg = len(y_train) - pos
    # Increase the loss contribution of rare positive labels relative to abundant negative examples.
    w = np.where(pos > 0, neg / np.maximum(pos, 1), 1.0)
    return torch.tensor(w, dtype=torch.float32)


def compute_binary_pos_weight(y_train_col):
    """
    Compute the positive class weight for one binary label in the One-vs-Rest setting.
    """
    pos = float(y_train_col.sum())
    neg = float(len(y_train_col) - pos)
    return torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32)


def metrics_from_binary_matrix(y_true, y_pred):
    """
    Compute aggregate evaluation metrics from binary multi-label predictions.

    The function returns micro F1, macro F1, and Hamming loss to capture
    complementary aspects of multi-label prediction quality.
    """
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    ham = hamming_loss(y_true, y_pred)
    return {
        "micro_f1": float(micro_f1),
        "macro_f1": float(macro_f1),
        "hamming_loss": float(ham),
    }


def choose_thresholds(y_true, probs, labels, mode="f1", min_precision=0.50, default_threshold=0.5):
    """
    Select one decision threshold per label based on validation probabilities.

    In F1 mode, thresholds are chosen to maximize per-label F1. In dashboard
    mode, the search prioritizes thresholds that satisfy a minimum precision
    requirement before considering recall and F1 trade-offs.
    """
    chosen = []
    
    # Allow selected business-critical labels to use a more permissive threshold strategy so they remain visible in dashboard outputs even when they are hard to detect.
    business_critical_labels = ['document_management', 'smarthealth_epa_features']
    
    for j, label in enumerate(labels):
        best = {
            "label": label,
            "threshold": default_threshold,
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "mode": mode,
        }
        candidates = []
        # Evaluate a grid of candidate thresholds instead of assuming that 0.5 is optimal for every label.
        for th in THRESHOLD_GRID:
            pred = (probs[:, j] >= th).astype(int)
            p, r, f, _ = precision_recall_fscore_support(
                y_true[:, j], pred, average="binary", zero_division=0
            )
            candidates.append({
                "threshold": float(th), "precision": float(p), "recall": float(r), "f1": float(f)
            })
            
        if mode == "f1":
            best_c = max(candidates, key=lambda x: (x["f1"], -x["threshold"]))
            
        elif mode == "dashboard":
            # --- Hybrid Optimization ---
            if label in business_critical_labels:
                # Optimize for F1 without strict precision constraints to ensure dashboard visibility
                best_c = max(candidates, key=lambda x: (x["f1"], -x["threshold"]))
                
                # Safety net: prevent threshold from dropping into extreme noise (< 0.25)
                if best_c["threshold"] < 0.25:
                    fallback = [c for c in candidates if c["threshold"] >= 0.25]
                    if fallback:
                        best_c = fallback[0] # Apply the 0.25 floor
            else:
                # Standard logic: require minimum precision
                ok = [c for c in candidates if c["precision"] >= min_precision]
                if ok:
                    # If precision is met, maximize F1
                    best_c = max(ok, key=lambda x: (x["f1"], x["recall"], -x["threshold"]))
                else:
                    # Fallback if no threshold meets precision
                    best_c = max(candidates, key=lambda x: (x["precision"], x["f1"], x["threshold"]))
        else:
            raise ValueError("mode must be 'f1' or 'dashboard'")
            
        best.update(best_c)
        chosen.append(best)
        
    return pd.DataFrame(chosen)


def apply_thresholds(probs, threshold_df, labels):
    """
    Apply label-specific thresholds to a probability matrix and return binary predictions.
    """
    thresholds = threshold_df.set_index("label").loc[labels, "threshold"].to_numpy()
    return (probs >= thresholds.reshape(1, -1)).astype(int)

## Data preparation

The data loading procedure mirrors the earlier notebook structure so that the final experiment remains comparable to the prior workflow. The review text is normalized, the topic string is split into a list of labels, and a multi-hot target matrix is built for the joint multi-label setup. The same target matrix is also reused for the One-vs-Rest configuration by extracting one target column per label.

This shared preparation stage is important because it isolates the effect of modeling formulation and loss configuration. If each experimental branch relied on a different preprocessing path, differences in performance would be harder to interpret.

In [ ]:
# =========================
# LOAD DATA
# =========================

labeled_df = pd.read_excel(LABELED_PATH)
print("Loaded labeled rows:", len(labeled_df))
print("Columns:", list(labeled_df.columns))

# Resolve the text and ID columns dynamically so the notebook remains usable even if intermediate file schemas change slightly.
TEXT_COL = find_first_existing_column(labeled_df, TEXT_CANDIDATES)
ID_COL = find_first_existing_column(labeled_df, ID_CANDIDATES, required=False)
if ID_COL is None:
    ID_COL = "row_id"
    labeled_df[ID_COL] = [f"labeled_{i}" for i in range(len(labeled_df))]

labeled_df[ID_COL] = labeled_df[ID_COL].astype(str).str.strip()
labeled_df[TEXT_COL] = labeled_df[TEXT_COL].apply(normalize_text)
labeled_df[TOPIC_COL] = labeled_df[TOPIC_COL].apply(clean_topic_string)

# Remove rows without usable text or labels because they cannot contribute meaningful supervision.
labeled_df = labeled_df[labeled_df[TEXT_COL] != ""].copy().reset_index(drop=True)
labeled_df["topics"] = labeled_df[TOPIC_COL].apply(split_topics)
labeled_df = labeled_df[labeled_df["topics"].map(len) > 0].copy().reset_index(drop=True)

# Derive one shared global label space so all experimental variants are trained and evaluated on identical targets.
all_labels = sorted({t for topics in labeled_df["topics"] for t in topics})
label2id = {label: i for i, label in enumerate(all_labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(all_labels)

labeled_df["targets"] = labeled_df["topics"].apply(lambda x: build_multihot(x, label2id))
X_text = labeled_df[TEXT_COL].tolist()
Y = np.stack(labeled_df["targets"].values)

print("Text column:", TEXT_COL)
print("ID column:", ID_COL)
print("Number of labels:", num_labels)
print(all_labels)
print("Average labels per review:", round(Y.sum(axis=1).mean(), 2))
display(pd.Series(np.sum(Y, axis=0), index=all_labels).sort_values(ascending=False).to_frame("count"))

## Training cell

This cell trains all experimental variants within one unified loop:

- `joint_multilabel` with `bce`
- `joint_multilabel` with `weighted_bce`
- `one_vs_rest` with `bce`
- `one_vs_rest` with `weighted_bce`

The training stage stores fold-level probabilities and true labels rather than committing to one thresholding choice immediately. This is intentional. Thresholding is treated as a separate decision layer because the thesis question distinguishes between model-centric performance and dashboard-centric utility. Separating training from threshold selection makes that distinction explicit.

The previous runtime error was caused by a mismatch between pretrained classifier head dimensions and task-specific output dimensions. That issue is corrected in this notebook by loading models with `ignore_mismatched_sizes=True`, which reinitializes the incompatible classification head while keeping the pretrained backbone.

In [ ]:
# =========================
# TRAINING CELL
# =========================

all_training_outputs = {}
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
fold_splits = list(kf.split(X_text))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

for approach in APPROACHES:
    for loss_mode in LOSS_MODES:
        run_key = f"{approach}__{loss_mode}"
        print("=" * 100)
        print(f"Running setup: {run_key}")
        print("=" * 100)
        # Store fold-level outputs so probabilities from all validation folds can later be pooled for threshold selection.
        fold_outputs = []

        for fold_idx, (train_idx, val_idx) in enumerate(fold_splits, start=1):
            print(f"Fold {fold_idx}/{NUM_FOLDS}")
            train_texts = [X_text[i] for i in train_idx]
            val_texts = [X_text[i] for i in val_idx]
            y_train = Y[train_idx]
            y_val = Y[val_idx]

            # Train one shared model that predicts all topic labels simultaneously.
            if approach == "joint_multilabel":
                train_ds = MultiLabelTextDataset(train_texts, y_train, tokenizer, MAX_LENGTH)
                val_ds = MultiLabelTextDataset(val_texts, y_val, tokenizer, MAX_LENGTH)
                pos_weight = compute_pos_weight(y_train) if loss_mode == "weighted_bce" else None
                model = make_joint_model(MODEL_NAME, num_labels, id2label, label2id)
                fold_dir = OUTPUT_DIR / run_key / f"fold_{fold_idx}"
                args = TrainingArguments(
                    output_dir=str(fold_dir),
                    learning_rate=LEARNING_RATE,
                    per_device_train_batch_size=TRAIN_BATCH_SIZE,
                    per_device_eval_batch_size=EVAL_BATCH_SIZE,
                    num_train_epochs=NUM_EPOCHS,
                    weight_decay=WEIGHT_DECAY,
                    eval_strategy="epoch",
                    save_strategy="no",
                    logging_strategy="epoch",
                    report_to="none",
                    seed=SEED + fold_idx,
                    remove_unused_columns=False,
                )
                trainer = BCETrainer(
                    model=model,
                    args=args,
                    train_dataset=train_ds,
                    eval_dataset=val_ds,
                    pos_weight=pos_weight,
                )
                trainer.train()
                pred = trainer.predict(val_ds)
                logits = pred.predictions
                probs = sigmoid(logits)
                fold_outputs.append({
                    "fold": fold_idx,
                    "val_idx": val_idx.tolist(),
                    "y_true": y_val,
                    "logits": logits,
                    "probs": probs,
                })
                if SAVE_FOLD_MODELS:
                    trainer.save_model(str(fold_dir / "final_model"))
                del model, trainer
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            # Train one separate binary classifier per label and combine their predictions into one matrix.
            elif approach == "one_vs_rest":
                logits_matrix = np.zeros_like(y_val, dtype=float)
                probs_matrix = np.zeros_like(y_val, dtype=float)
                for j, label in enumerate(all_labels):
                    y_train_j = y_train[:, j:j+1]
                    y_val_j = y_val[:, j:j+1]
                    train_ds = MultiLabelTextDataset(train_texts, y_train_j, tokenizer, MAX_LENGTH)
                    val_ds = MultiLabelTextDataset(val_texts, y_val_j, tokenizer, MAX_LENGTH)
                    pos_weight = compute_binary_pos_weight(y_train_j[:, 0]) if loss_mode == "weighted_bce" else None
                    model = make_binary_model(MODEL_NAME, label)
                    fold_dir = OUTPUT_DIR / run_key / f"fold_{fold_idx}" / label
                    args = TrainingArguments(
                        output_dir=str(fold_dir),
                        learning_rate=LEARNING_RATE,
                        per_device_train_batch_size=TRAIN_BATCH_SIZE,
                        per_device_eval_batch_size=EVAL_BATCH_SIZE,
                        num_train_epochs=NUM_EPOCHS,
                        weight_decay=WEIGHT_DECAY,
                        eval_strategy="no",
                        save_strategy="no",
                        logging_strategy="no",
                        report_to="none",
                        seed=SEED + fold_idx,
                        remove_unused_columns=False,
                    )
                    trainer = BCETrainer(
                        model=model,
                        args=args,
                        train_dataset=train_ds,
                        eval_dataset=val_ds,
                        pos_weight=pos_weight,
                    )
                    trainer.train()
                    
                    # Keep raw validation predictions so thresholding can be decided later rather than hard-coded during training.
                    pred = trainer.predict(val_ds)
                    logits = pred.predictions.reshape(-1)
                    logits_matrix[:, j] = logits
                    probs_matrix[:, j] = sigmoid(logits)
                    if SAVE_FOLD_MODELS:
                        trainer.save_model(str(fold_dir / "final_model"))
                    del model, trainer
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                fold_outputs.append({
                    "fold": fold_idx,
                    "val_idx": val_idx.tolist(),
                    "y_true": y_val,
                    "logits": logits_matrix,
                    "probs": probs_matrix,
                })
            else:
                raise ValueError("Unknown approach")

        all_training_outputs[run_key] = fold_outputs

print("Training finished.")

## Evaluation cell

The evaluation cell applies two thresholding strategies to each trained setup.

### F1 threshold mode

This threshold mode mirrors the earlier workflow. It selects the threshold that maximizes per-label F1 on the pooled validation predictions.

### Dashboard threshold mode

This threshold mode is intended for operational use. The procedure first searches for thresholds that reach a minimum precision target. Among those thresholds, it prefers higher recall and then better F1. If no threshold reaches the target precision, the fallback is the threshold that maximizes precision and then F1. This prevents the procedure from defaulting too easily to very permissive thresholds that inflate recall.

The evaluation reports three main aggregate metrics:

- **Micro F1**, which emphasizes overall instance-level label decisions.
- **Macro F1**, which gives equal importance to each label.
- **Hamming Loss**, which quantifies the share of incorrect binary label decisions across the full prediction matrix.

At the end of evaluation, the best setup under the dashboard threshold mode is selected and marked for final export.

In [ ]:
# =========================
# EVALUATION CELL
# =========================

summary_rows = []
threshold_rows = []
per_label_rows = []

for run_key, fold_outputs in all_training_outputs.items():
    approach, loss_mode = run_key.split("__")
    
    # Pool validation predictions across folds to obtain one combined basis for threshold tuning and final comparison.
    all_y_true = np.vstack([x["y_true"] for x in fold_outputs])
    all_probs = np.vstack([x["probs"] for x in fold_outputs])

    # Evaluate each trained setup under both model-centric and dashboard-centric threshold strategies.
    for threshold_mode in ["f1", "dashboard"]:
        threshold_df = choose_thresholds(
            all_y_true,
            all_probs,
            labels=all_labels,
            mode=threshold_mode,
            min_precision=DASHBOARD_MIN_PRECISION,
            default_threshold=DEFAULT_THRESHOLD,
        )
        y_pred = apply_thresholds(all_probs, threshold_df, all_labels)
        metrics = metrics_from_binary_matrix(all_y_true, y_pred)
        report = classification_report(
            all_y_true,
            y_pred,
            target_names=all_labels,
            output_dict=True,
            zero_division=0,
        )

        # Store aggregate metrics for ranking the experimental variants across approaches, loss modes, and threshold modes.
        summary_rows.append({
            "model_name": MODEL_NAME,
            "approach": approach,
            "loss_mode": loss_mode,
            "threshold_mode": threshold_mode,
            **metrics,
        })

        tdf = threshold_df.copy()
        tdf["model_name"] = MODEL_NAME
        tdf["approach"] = approach
        tdf["loss_mode"] = loss_mode
        tdf["threshold_mode"] = threshold_mode
        threshold_rows.append(tdf)

        for label in all_labels:
            per_label_rows.append({
                "model_name": MODEL_NAME,
                "approach": approach,
                "loss_mode": loss_mode,
                "threshold_mode": threshold_mode,
                "label": label,
                "precision": report[label]["precision"],
                "recall": report[label]["recall"],
                "f1": report[label]["f1-score"],
                "support": report[label]["support"],
            })

summary_df = pd.DataFrame(summary_rows)
thresholds_df = pd.concat(threshold_rows, ignore_index=True)
per_label_df = pd.DataFrame(per_label_rows)

summary_df = summary_df.sort_values(
    by=["threshold_mode", FINAL_SELECTION_METRIC, "micro_f1"],
    ascending=[True, False, False],
).reset_index(drop=True)

summary_path = OUTPUT_DIR / "summary_metrics.csv"
thresholds_path = OUTPUT_DIR / "thresholds_all_modes.csv"
per_label_path = OUTPUT_DIR / "per_label_report.csv"

summary_df.to_csv(summary_path, index=False)
thresholds_df.to_csv(thresholds_path, index=False)
per_label_df.to_csv(per_label_path, index=False)

print("Summary metrics")
display(summary_df)
print("\nThreshold preview")
display(thresholds_df.head(64))
print("\nPer-label preview")
display(per_label_df.head(64))
print("\nSaved files:")
print(summary_path)
print(thresholds_path)
print(per_label_path)

# Select the final candidate according to the predefined dashboard-oriented model selection rule.
best_dashboard_row = (
    summary_df[summary_df["threshold_mode"] == FINAL_SELECTION_THRESHOLD_MODE]
    .sort_values(by=[FINAL_SELECTION_METRIC, "micro_f1", "hamming_loss"], ascending=[False, False, True])
    .iloc[0]
)
print("\nSelected best dashboard setup:")
print(best_dashboard_row.to_dict())

#### Model selection result

The summary tables produced above allow the final experimental variants to be compared under both threshold modes. The setup selected for export is the one that performs best under the dashboard-oriented threshold strategy according to the predefined final selection metric.

## Final model export

Cross-validation is appropriate for model comparison, but the dashboard ultimately needs one deployable model. Therefore, once the best experimental setup has been identified, the notebook retrains that exact setup on the full labeled dataset and exports the final model artifacts.

This final export stage performs three tasks:

1. Retrains the selected modeling formulation on the full dataset.
2. Recomputes thresholds on the full dataset predictions for deployment use.
3. Saves model files and threshold configuration into a dedicated export directory.

This export is intended as the final candidate for dashboard integration. The threshold mode used for export follows the dashboard-oriented selection logic defined earlier.

In [ ]:
# =========================
# FINAL EXPORT CELL
# =========================

if EXPORT_FINAL_BEST_MODEL:
    best_approach = best_dashboard_row["approach"]
    best_loss_mode = best_dashboard_row["loss_mode"]
    export_dir = OUTPUT_DIR / "best_dashboard_model"
    export_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    full_texts = X_text
    full_y = Y

    # Retrain the selected joint model on the full labeled dataset so one deployable final model is produced.
    if best_approach == "joint_multilabel":
        full_ds = MultiLabelTextDataset(full_texts, full_y, tokenizer, MAX_LENGTH)
        pos_weight = compute_pos_weight(full_y) if best_loss_mode == "weighted_bce" else None
        model = make_joint_model(MODEL_NAME, num_labels, id2label, label2id)
        args = TrainingArguments(
            output_dir=str(export_dir / "joint_multilabel_training"),
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            num_train_epochs=NUM_EPOCHS,
            weight_decay=WEIGHT_DECAY,
            eval_strategy="no",
            save_strategy="no",
            logging_strategy="epoch",
            report_to="none",
            seed=SEED,
            remove_unused_columns=False,
        )
        trainer = BCETrainer(
            model=model,
            args=args,
            train_dataset=full_ds,
            eval_dataset=full_ds,
            pos_weight=pos_weight,
        )
        trainer.train()
        pred = trainer.predict(full_ds)
        full_probs = sigmoid(pred.predictions)
        final_threshold_df = choose_thresholds(
            full_y,
            full_probs,
            labels=all_labels,
            mode=FINAL_SELECTION_THRESHOLD_MODE,
            min_precision=DASHBOARD_MIN_PRECISION,
            default_threshold=DEFAULT_THRESHOLD,
        )
        trainer.save_model(str(export_dir / "model"))
        tokenizer.save_pretrained(str(export_dir / "model"))

    # Retrain one binary model per label when the One-vs-Rest formulation is selected as the best final approach.
    elif best_approach == "one_vs_rest":
        ovr_dir = export_dir / "ovr_models"
        ovr_dir.mkdir(parents=True, exist_ok=True)
        full_probs = np.zeros_like(full_y, dtype=float)
        for j, label in enumerate(all_labels):
            y_col = full_y[:, j:j+1]
            full_ds = MultiLabelTextDataset(full_texts, y_col, tokenizer, MAX_LENGTH)
            pos_weight = compute_binary_pos_weight(y_col[:, 0]) if best_loss_mode == "weighted_bce" else None
            model = make_binary_model(MODEL_NAME, label)
            args = TrainingArguments(
                output_dir=str(ovr_dir / f"training_{label}"),
                learning_rate=LEARNING_RATE,
                per_device_train_batch_size=TRAIN_BATCH_SIZE,
                per_device_eval_batch_size=EVAL_BATCH_SIZE,
                num_train_epochs=NUM_EPOCHS,
                weight_decay=WEIGHT_DECAY,
                eval_strategy="no",
                save_strategy="no",
                logging_strategy="no",
                report_to="none",
                seed=SEED,
                remove_unused_columns=False,
            )
            trainer = BCETrainer(
                model=model,
                args=args,
                train_dataset=full_ds,
                eval_dataset=full_ds,
                pos_weight=pos_weight,
            )
            trainer.train()
            pred = trainer.predict(full_ds)
            full_probs[:, j] = sigmoid(pred.predictions.reshape(-1))
            label_dir = ovr_dir / label
            trainer.save_model(str(label_dir))
            tokenizer.save_pretrained(str(label_dir))
            del model, trainer
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        # Recompute deployment thresholds on the full labeled dataset so the exported model package includes its operating rule.
        final_threshold_df = choose_thresholds(
            full_y,
            full_probs,
            labels=all_labels,
            mode=FINAL_SELECTION_THRESHOLD_MODE,
            min_precision=DASHBOARD_MIN_PRECISION,
            default_threshold=DEFAULT_THRESHOLD,
        )
    else:
        raise ValueError("Unknown final approach")

    final_pred = apply_thresholds(full_probs, final_threshold_df, all_labels)
    final_metrics = metrics_from_binary_matrix(full_y, final_pred)

    # Save the final selection metadata together with the exported model so the deployment package remains self-describing.
    export_meta = {
        "model_name": MODEL_NAME,
        "approach": best_approach,
        "loss_mode": best_loss_mode,
        "threshold_mode": FINAL_SELECTION_THRESHOLD_MODE,
        "selection_metric": FINAL_SELECTION_METRIC,
        "final_metrics_on_full_data": final_metrics,
        "labels": all_labels,
    }

    with open(export_dir / "model_selection_summary.json", "w", encoding="utf-8") as f:
        json.dump(export_meta, f, ensure_ascii=False, indent=2)

    final_threshold_df.to_csv(export_dir / "dashboard_thresholds.csv", index=False)

    print("Final dashboard model exported to:", export_dir)
    print(json.dumps(export_meta, ensure_ascii=False, indent=2))

#### Export interpretation

The exported final model is the deployment candidate that corresponds to the best-performing dashboard-oriented experimental setup. Its reported metrics are calculated on the full labeled dataset after retraining and should therefore be interpreted as deployment-oriented reference values rather than as an unbiased estimate of generalization performance.

## Final note on methodological scope

This notebook is designed as the final focused experiment for model selection rather than as an open-ended search across many architectures and hyperparameters. The goal is to compare a small number of methodologically meaningful alternatives in a transparent and reproducible way and to export one final model candidate for downstream dashboard integration.